# Random Nucleotide Sampling

Generate DNA/RNA sequence variants by filling masked positions with random bases from an IUPAC substitution scheme. Auto-detects DNA vs RNA and converts T to U as needed.

In [1]:
from proto_tools import (
    RandomNucleotideSampleConfig,
    RandomNucleotideSampleInput,
    run_random_nucleotide_sample,
)
from proto_tools.tools.masked_models.masking import MaskingStrategy

## 1. Basic usage with pre-masked DNA

Use `_` to mark positions for random substitution. Unmasked positions are preserved.

In [2]:
inputs = RandomNucleotideSampleInput(sequences=["ACGT_CGT_A"])
result = run_random_nucleotide_sample(inputs)

print(f"Input:  ACGT_CGT_A")
print(f"Output: {result.sequences[0]}")

Input:  ACGT_CGT_A
Output: ACGTGCGTAA


## 2. IUPAC substitution schemes

Control which bases are allowed at masked positions. `N` = any base, `R` = purines only (A/G), `Y` = pyrimidines only (C/T).

In [3]:
for scheme in ["N", "R", "Y", "S", "W"]:
    config = RandomNucleotideSampleConfig(substitution_scheme=scheme, seed=42)
    result = run_random_nucleotide_sample(
        RandomNucleotideSampleInput(sequences=["____"]),
        config,
    )
    print(f"{scheme}: {result.sequences[0]}")

N: AAGC
R: AAGA
Y: CCTC
S: GGCG
W: AATA


## 3. RNA auto-detection

When input contains U, the tool auto-detects RNA and converts sampled T bases to U.

In [4]:
rna_input = RandomNucleotideSampleInput(sequences=["AUGC_CGU"])
result = run_random_nucleotide_sample(rna_input)

print(f"Input:  AUGC_CGU")
print(f"Output: {result.sequences[0]}")
print(f"Contains U: {'U' in result.sequences[0]}")
print(f"Contains T: {'T' in result.sequences[0]}")

Input:  AUGC_CGU
Output: AUGCGCGU
Contains U: True
Contains T: False


## 4. Masking strategy with batch export

In [5]:
wild_type = "ACGTACGTAC"
# Duplicate to generate 5 independent variants
inputs = RandomNucleotideSampleInput(sequences=[wild_type] * 5)
config = RandomNucleotideSampleConfig(
    masking_strategy=MaskingStrategy(num_mutations=2),
)
result = run_random_nucleotide_sample(inputs, config)

for i, seq in enumerate(result.sequences):
    print(f"Variant {i}: {seq}")

# Export to FASTA
result.export(name="random_nucleotide_variants", export_path=".", file_format="fasta")

Variant 0: ACGTTCGGAC
Variant 1: ACGCACGTAC
Variant 2: ACGTACGTTC
Variant 3: ACGTACGGAC
Variant 4: GTGTACGTAC
